# 02 YOLO Object Detection

This session uses the camera with `yolo11n.pt`, the nano YOLO model.

We keep the model's default COCO-style labels so we can demo object detection without collecting a custom dataset first.

If a cell fails here, go back to `00_environment_check.ipynb` and `01_web_panel_intro.ipynb` first.


## Safety

This notebook does **not** drive the rover.

Keep the rover on a stand or disconnected from motor power if you are testing on the full robot. Start with a still scene, then try the short live preview.


In [ ]:
import glob
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import torch
from jetcar.camera import open_camera, read_rgb_frame
from jetcar.yolo import (
    annotated_frame,
    detection_lines,
    load_yolo_model,
    predict_frame,
    resolve_yolo_model_path,
)


In [ ]:
camera_source = 'usb'
sensor_id = 0
device_index = 0
frame_width = 1280
frame_height = 720
warmup_frames = 12

model_name = 'yolo11n.pt'
conf_threshold = 0.25
image_size = 640
max_detections = 20

live_frames = 20
live_pause_seconds = 0.15

print('Edit the values in this cell if needed, then run the next cells.')


In [ ]:
video_devices = sorted(glob.glob('/dev/video*'))
resolved_model = resolve_yolo_model_path(model_name, project_root=project_root)

print('Project root:', project_root)
print('Video devices:', video_devices or ['none found'])
print('CUDA available:', torch.cuda.is_available())
print('YOLO model path:', resolved_model)


In [ ]:
_yolo_model = load_yolo_model(model_name, project_root=project_root)
print('Loaded model:', resolve_yolo_model_path(model_name, project_root=project_root))
print('Task:', _yolo_model.task)
print('Class count:', len(_yolo_model.names))
print('First 10 classes:', [_yolo_model.names[i] for i in range(min(10, len(_yolo_model.names)))])


In [ ]:
cap = open_camera(
    source=camera_source,
    sensor_id=sensor_id,
    device_index=device_index,
    width=frame_width,
    height=frame_height,
    warmup_frames=warmup_frames,
)
try:
    frame = read_rgb_frame(cap)
finally:
    cap.release()

result = predict_frame(
    _yolo_model,
    frame,
    conf=conf_threshold,
    imgsz=image_size,
    max_det=max_detections,
)
overlay = annotated_frame(result)

print('Detections:')
for line in detection_lines(result):
    print('-', line)

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.imshow(frame)
plt.title('Camera Frame')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(overlay)
plt.title('YOLO Overlay')
plt.axis('off')
plt.tight_layout()
plt.show()


## Optional Live Preview

Run the next cell for a short live demo. Stop the cell if you want to exit early.


In [ ]:
import time
from IPython.display import clear_output

cap = open_camera(
    source=camera_source,
    sensor_id=sensor_id,
    device_index=device_index,
    width=frame_width,
    height=frame_height,
    warmup_frames=warmup_frames,
)
try:
    for frame_index in range(max(live_frames, 1)):
        frame = read_rgb_frame(cap)
        result = predict_frame(
            _yolo_model,
            frame,
            conf=conf_threshold,
            imgsz=image_size,
            max_det=max_detections,
        )
        overlay = annotated_frame(result)

        clear_output(wait=True)
        plt.figure(figsize=(10, 6))
        plt.imshow(overlay)
        plt.title(f'YOLO live preview frame {frame_index + 1}/{live_frames}')
        plt.axis('off')
        plt.show()
        for line in detection_lines(result):
            print('-', line)
        time.sleep(max(live_pause_seconds, 0.0))
finally:
    cap.release()
